In [103]:
import pandas as pd
import re
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split


# params
words_count = 5000 # number of words in a dictionary
max_msg_length = 100 # max. length (in words) for a single message
val_set_size = 0.1; test_set_size = 0.1 # sizes of validation and test sets (from 0 to 1)


# returns word : index dictionary from words included in data
def encode_words(data, vocab_size=5000):
    word_counts = Counter()

    for text in data['text']:
        words = re.findall(r'\w+', text)
        word_counts.update(words)
        
    top_words = word_counts.most_common(vocab_size)

    word2idx = {'<PAD>': 0, '<UNK>': 1}
    for idx, word in enumerate(top_words, start=2):
        word2idx[word[0]] = idx
        
    return word2idx


# returns a list of encoded messages using word : index dictionary 
def encode_messages(df, word2idx, max_len=100):
    encoded_messages = []
    
    for text in df['text']:
        encoded = [word2idx.get(word, word2idx['<UNK>']) for word in re.findall(r'\w+', text)][:max_len]
        
        if len(encoded) < max_len:
            encoded += ([word2idx['<PAD>']] * (max_len - len(encoded)))
        encoded_messages.append(encoded)
    
    return encoded_messages
    

# getting the data from csv file and 
df_data = pd.read_csv('../data/spam.csv', encoding='latin-1', usecols=[0, 1])      # get data from the file
df_data.columns = ['label', 'text']                                                # rename columns
df_data['label'] = df_data['label'].map({'ham': 0, 'spam': 1})                     # set ham/spam to 0/1
df_data['text'] = df_data['text'].apply(lambda x: x.lower())                       # make all letters lowercase

# split the dataset into train, validation and test sets 
df_train_val, df_test = train_test_split(
    df_data, 
    test_size=test_set_size, 
    random_state=42, 
    stratify=df_data['label']
)
df_train, df_val = train_test_split(
    df_train_val, 
    test_size=val_set_size/(1-test_set_size), 
    random_state=42, 
    stratify=df_train_val['label']
)

# encode datasets (train, validation, test)
word2idx = encode_words(df_train, words_count)
X_train = encode_messages(df_train, word2idx, max_msg_length) 
X_val = encode_messages(df_val, word2idx, max_msg_length) 
X_test = encode_messages(df_test, word2idx, max_msg_length) 
y_train = df_train['label'].values
y_val = df_val['label'].values
y_test = df_test['label'].values